# LLM Text Preprocessing and Embeddings

## 1. Introduction & Data Loading
**Why this matters for LLMs / Agentic Systems:**
Large Language Models cannot process raw text directly. They require numerical input. The first step in building any LLM or agentic system is loading the raw text data and preparing it for transformation. This step is crucial because the quality and format of the raw data dictate the foundation upon which the model learns. If the text is noisy or improperly loaded, the subsequent tokenization and embedding steps will be flawed, leading to poor model performance.

In [ ]:
import urllib.request

url = ("https://raw.githubusercontent.com/rasbt/"
       "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:99])

## 2. Tokenization
**Why this matters for LLMs / Agentic Systems:**
Tokenization is the process of breaking down raw text into smaller, manageable units called tokens (which can be words, subwords, or characters). Modern LLMs use subword tokenization (like BPE, Byte Pair Encoding, used by `tiktoken`). This is vital because it strikes a balance between vocabulary size and the ability to handle out-of-vocabulary words. By breaking unknown words into known subwords, the model can infer meaning from morphological structures (e.g., "un-break-able"), making the agent more robust when encountering novel text in the wild.

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
encoded_text = tokenizer.encode(raw_text)

print("Total number of tokens:", len(encoded_text))
print("First 10 token IDs:", encoded_text[:10])
print("Decoded back:", tokenizer.decode(encoded_text[:10]))

## 3. Data Sampling with Sliding Window
**Why this matters for LLMs / Agentic Systems:**
To train an LLM to predict the next word, we need to create input-target pairs from our tokenized text. We use a sliding window approach to extract sequences of a fixed `max_length`. The `stride` determines how many positions we shift the window to get the next sample. 

**Experiment: Changing `max_length` and `stride`**
Below, we experiment with different `stride` values. Overlap (when `stride < max_length`) is highly useful because it provides the model with diverse contexts for the same words. It ensures that the model learns to predict the next word across different sequence boundaries, improving generalization and data efficiency.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    return dataloader

# --- EXPERIMENT ---
# 1. Original settings (no overlap)
dataloader_orig = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
print("Original (max_length=4, stride=4) - Total samples:", len(dataloader_orig.dataset))

# 2. Experiment settings (with overlap)
dataloader_exp = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=2, shuffle=False)
print("Experiment (max_length=4, stride=2) - Total samples:", len(dataloader_exp.dataset))

print("\nNotice how decreasing the stride (increasing overlap) significantly increases the number of training samples we can extract from the same text.")

## 4. Creating Token and Positional Embeddings
**Why this matters for LLMs / Agentic Systems:**
Once we have token IDs, we must convert them into continuous vector representations (embeddings) before feeding them into the neural network. We also add positional embeddings because the self-attention mechanism in Transformers has no inherent notion of word order.

**Why do embeddings encode meaning, and how are they related to NN concepts?**
Embeddings encode meaning by mapping discrete tokens into a continuous, high-dimensional vector space. Instead of treating words as isolated, orthogonal categories (like in one-hot encoding), embeddings place words with similar semantic meanings closer together in this space. 

This is deeply related to the core Neural Network concept of **representation learning**. During the training process (via backpropagation), the embedding weights are continuously adjusted to minimize the loss function (e.g., predicting the next word). Consequently, words that appear in similar contexts are forced to have similar vector representations. This allows the LLM to generalize its understanding—if it learns a concept about "dog", the continuous nature of embeddings helps it apply similar logic to "cat" or "puppy".

In [ ]:
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

# Get a batch from our original dataloader
data_iter = iter(dataloader_orig)
inputs, targets = next(data_iter)

print("Token IDs shape:", inputs.shape)
token_embeddings = token_embedding_layer(inputs)
print("Token Embeddings shape:", token_embeddings.shape)

# Positional embeddings
context_length = 4
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print("Positional Embeddings shape:", pos_embeddings.shape)

# Final Input embeddings (Token + Positional)
input_embeddings = token_embeddings + pos_embeddings
print("Final Input Embeddings shape:", input_embeddings.shape)